In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import scipy.stats as stats
from scipy.stats import spearmanr, hypergeom

from statsmodels.stats.multitest import multipletests

# GO enrichment
import gseapy as gp

## CellOracle
import celloracle as co

## Load data

-> processed with noisy clusters filtered

-> including base GRN

In [2]:
# --- LOAD AnnData ---
adata = sc.read_h5ad("../data/data_diff_express_lncRNA.h5ad")

## CellOracle base GRN for mouse 
#base_GRN_raw = pd.read_parquet("../data/celloracle_data/TFinfo_data/mm9_mouse_atac_atlas_data_TSS_and_cicero_0.9_accum_threshold_10.5_DF_peaks_by_TFs_v202204.parquet")

# Already in edge_list:
base_GRN = pd.read_parquet('base_GRN_edge_list.parquet')

Original extraction of base GRN data

In [3]:
'''
print(co.data.__dict__.keys())  # explore available functions
base_GRN = co.data.load_mouse_scATAC_atlas_base_GRN()
print(base_GRN.shape)
print(base_GRN.head(10))
print(base_GRN.columns.tolist())
'''

'\nprint(co.data.__dict__.keys())  # explore available functions\nbase_GRN = co.data.load_mouse_scATAC_atlas_base_GRN()\nprint(base_GRN.shape)\nprint(base_GRN.head(10))\nprint(base_GRN.columns.tolist())\n'

Structure inspection of raw base GRN:

In [4]:
'''
# =============================================================
# UNDERSTAND THE STRUCTURE OF RAW BASE GRN 
# =============================================================

print(base_GRN_raw.shape)
print(base_GRN_raw.head(10))
print(base_GRN_raw.columns.tolist())

print("    ")
print("    ")

# peak_id, gene_short_name = metadata columns
# remaining columns = one per TF, binary (1 = motif found in peak)

tf_columns = [c for c in base_GRN_raw.columns if c not in ['peak_id', 'gene_short_name']]
print(f"Number of peaks (rows): {len(base_GRN_raw)}")
print(f"Number of TFs (columns): {len(tf_columns)}")
'''

'\n# =============================================================\n# UNDERSTAND THE STRUCTURE OF RAW BASE GRN \n# =============================================================\n\nprint(base_GRN_raw.shape)\nprint(base_GRN_raw.head(10))\nprint(base_GRN_raw.columns.tolist())\n\nprint("    ")\nprint("    ")\n\n# peak_id, gene_short_name = metadata columns\n# remaining columns = one per TF, binary (1 = motif found in peak)\n\ntf_columns = [c for c in base_GRN_raw.columns if c not in [\'peak_id\', \'gene_short_name\']]\nprint(f"Number of peaks (rows): {len(base_GRN_raw)}")\nprint(f"Number of TFs (columns): {len(tf_columns)}")\n'

Transforming to an standard edge list:

In [5]:
'''
# =============================================================
# CONVERT TO EDGE LIST (source -> target)
# =============================================================
# For each peak, for each TF with motif present (value == 1),
# create a directed edge: TF -> gene_short_name
# This is the format CellOracle uses internally for the prior network.

def base_GRN_to_edge_list(base_GRN_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Converts the CellOracle peak x TF matrix into a simple directed edge list.
    
    Each row in the output represents: TF (source) can regulate gene (target)
    because the TF motif was found in an open chromatin peak near that gene.
    """
    tf_cols = [c for c in base_GRN_raw.columns if c not in ['peak_id', 'gene_short_name']]
    
    edges = []
    for _, row in base_GRN_raw.iterrows():
        target_gene = row['gene_short_name']
        # Only keep TFs with motif present in this peak
        active_tfs = [tf for tf in tf_cols if row[tf] == 1.0]
        for tf in active_tfs:
            edges.append({'source': tf, 'target': target_gene})
    
    edge_df = pd.DataFrame(edges).drop_duplicates(subset=['source', 'target'])
    print(f"Total unique edges in base GRN: {len(edge_df)}")
    print(f"Unique TFs (sources): {edge_df['source'].nunique()}")
    print(f"Unique target genes: {edge_df['target'].nunique()}")
    return edge_df

## Convert
base_GRN = base_GRN_to_edge_list(base_GRN_raw)
print(base_GRN.head(10))

## Save the edge
base_GRN.to_parquet('base_GRN_edge_list.parquet', index=False)
'''

'\n# =============================================================\n# CONVERT TO EDGE LIST (source -> target)\n# =============================================================\n# For each peak, for each TF with motif present (value == 1),\n# create a directed edge: TF -> gene_short_name\n# This is the format CellOracle uses internally for the prior network.\n\ndef base_GRN_to_edge_list(base_GRN_raw: pd.DataFrame) -> pd.DataFrame:\n    """\n    Converts the CellOracle peak x TF matrix into a simple directed edge list.\n    \n    Each row in the output represents: TF (source) can regulate gene (target)\n    because the TF motif was found in an open chromatin peak near that gene.\n    """\n    tf_cols = [c for c in base_GRN_raw.columns if c not in [\'peak_id\', \'gene_short_name\']]\n    \n    edges = []\n    for _, row in base_GRN_raw.iterrows():\n        target_gene = row[\'gene_short_name\']\n        # Only keep TFs with motif present in this peak\n        active_tfs = [tf for tf in tf_

Structure of the base GRN edge list:

In [6]:
print(f"Total edges in base GRN: {len(base_GRN)}")
print(f"TFs (sources): {base_GRN['source'].nunique()}")
print(f"Target genes: {base_GRN['target'].nunique()}")

print("    ")
print("    ")

print(base_GRN.head(10))

Total edges in base GRN: 6876574
TFs (sources): 1093
Target genes: 21159
    
    
       source         target
0  Ac012531.1  4930430F08Rik
1        Atf3  4930430F08Rik
2      Bclaf1  4930430F08Rik
3     Bhlhe40  4930430F08Rik
4     Creb3l1  4930430F08Rik
5     Creb3l2  4930430F08Rik
6     Creb3l3  4930430F08Rik
7        E2f1  4930430F08Rik
8        E2f4  4930430F08Rik
9        E2f5  4930430F08Rik


## Differential expression ASO perturbed cluster VS control

-> we need this information to establish the connections of RMST1 and C13 lncRNA in the base GRN in CellOracle

-> necessary to consider the differences between these two clusters to only get the causality of the ASO

-> we compared neg. control cells in exto. clusters (1/3) VS ASO protocol cells per lncRNA in the perturbed cluster (5)

In [7]:
# Inicialize new .obs column for protocol + cluster information
adata.obs['phenotype_state'] = 'Other'

## Mask for neg. control cells in ecto. clusters
mask_ecto_control = (adata.obs['leiden'].isin(['1', '3'])) & \
                       (adata.obs['diff_protocol'] == 'mES_ectodiff_asoNegControl')

## Mask for neg. control cells in ecto. clusters
mask_ecto_unperturbed = (adata.obs['leiden'].isin(['1', '3'])) & \
                       (adata.obs['diff_protocol'] == 'mES_ectodiff')

## Mask for C13 in perturbed cluster
mask_c13_perturbed = (adata.obs['leiden'] == '5') & \
                    (adata.obs['diff_protocol'] == 'mES_ectodiff_asoC13')

## Mask for RMST1 in perturbed cluster
mask_rmst1_perturbed = (adata.obs['leiden'] == '5') & \
                      (adata.obs['diff_protocol'] == 'mES_ectodiff_asoRMST1')

# Save the metadata
adata.obs.loc[mask_ecto_control, 'phenotype_state'] = 'Control_Ecto'
adata.obs.loc[mask_ecto_unperturbed, 'phenotype_state'] = 'Unperturbed_Ecto'
adata.obs.loc[mask_c13_perturbed, 'phenotype_state'] = 'C13_perturbation'
adata.obs.loc[mask_rmst1_perturbed, 'phenotype_state'] = 'RMST1_perturbation'

## We work only with the defined phenotypes for the DE analysis
adata_pheno = adata[(adata.obs['phenotype_state'] != 'Other') & 
                     (adata.obs['phenotype_state'] != 'Unperturbed_Ecto')].copy()

print("Cells per phenotype:")
print(adata_pheno.obs['phenotype_state'].value_counts())

print("   ")
print("   ")

## DE analysis: each phenotype vs. ecto. control
sc.tl.rank_genes_groups(
    adata_pheno,
    groupby='phenotype_state',
    reference='Control_Ecto', 
    method='wilcoxon',      
    corr_method='benjamini-hochberg',
    use_raw=True, 
    key_added='DE_ASO_vs_Control'
)

## Extracting DE between control and each perturbation
de_c13 = sc.get.rank_genes_groups_df(adata_pheno, group='C13_perturbation', key='DE_ASO_vs_Control')
de_rmst1 = sc.get.rank_genes_groups_df(adata_pheno, group='RMST1_perturbation', key='DE_ASO_vs_Control')

# --- APPLY FILTERS ---
# 1. Protein coding only
# 2. Significant (p_adj < 0.01)
# 3. High Amplitude (LogFC > 1.5)
# 4. Remove Ribosomic/Mitochondrial genes

noise_prefixes = ('Rps', 'Rpl', 'mt-', 'MT-')
biotype_map = adata.raw.var['is_protein_coding'].fillna(False)

def significant_DE(de_df, biotype_map, lfc_thresh=1.5, padj_thresh=0.01):
    """
    Filter DE result DataFrame into upregulated and downregulated gene lists.
    Args:
        de_df: raw output from sc.get.rank_genes_groups_df
        biotype_map: Series mapping gene names to is_protein_coding boolean
        lfc_thresh: absolute logFC threshold (applied symmetrically)
        padj_thresh: adjusted p-value threshold
    Returns:
        (genes_up, genes_down): two lists of gene names
    """
    df = de_df.copy()
    df['is_protein_coding'] = df['names'].map(biotype_map).fillna(False)

    # Base quality filters
    mask_base = (
        (df['is_protein_coding'] == True) &
        (df['pvals_adj'] < padj_thresh) &
        (~df['names'].str.startswith(noise_prefixes))
    )

    genes_up = (
        df[mask_base & (df['logfoldchanges'] > lfc_thresh)]
        .sort_values('logfoldchanges', ascending=False)
        ['names'].tolist()
    )
    genes_down = (
        df[mask_base & (df['logfoldchanges'] < -lfc_thresh)]
        .sort_values('logfoldchanges', ascending=True)  # most negative first
        ['names'].tolist()
    )

    return genes_up, genes_down


# de_c13 and de_rmst1 are the FULL unfiltered DE DataFrames from rank_genes_groups_df
c13_up, c13_down     = significant_DE(de_c13, biotype_map)
rmst1_up, rmst1_down = significant_DE(de_rmst1, biotype_map)

print(f"C13   — UP: {len(c13_up)}, DOWN: {len(c13_down)}")
print(f"RMST1 — UP: {len(rmst1_up)}, DOWN: {len(rmst1_down)}")

... storing 'phenotype_state' as categorical


Cells per phenotype:
Control_Ecto          102
C13_perturbation       77
RMST1_perturbation     72
Name: phenotype_state, dtype: int64
   
   
C13   — UP: 60, DOWN: 689
RMST1 — UP: 30, DOWN: 706


GO ontology of DE genes with control reference to check how the perturbation biologically affects the cell

In [8]:
# =============================================================
# GO ENRICHMENT PER PERTURBATION AND DIRECTION
# =============================================================

def run_go_enrichment(gene_list: list, label: str, min_genes: int = 5) -> pd.DataFrame | None:
    """
    Runs gseapy Enrichr on a gene list and prints significant GO terms.
    Returns the full results DataFrame, or None if not enough genes.

    Args:
        gene_list: list of gene symbols
        label: descriptive label for print output
        min_genes: minimum number of genes required to run enrichment
    """
    print(f"\n{'='*55}")
    print(f" GO: {label}  ({len(gene_list)} genes)")
    print(f"{'='*55}")

    if len(gene_list) < min_genes:
        print(f"  Not enough genes (min={min_genes}). Skipping.")
        return None

    enrichment = gp.enrichr(
        gene_list=gene_list,
        gene_sets='GO_Biological_Process_2021',
        organism='mouse',
        outdir=None,
        no_plot=True
    )

    results = enrichment.results.sort_values('Adjusted P-value')
    sig_terms = results[results['Adjusted P-value'] < 0.05]

    if len(sig_terms) == 0:
        print("  No significant terms found (padj < 0.05).")
    else:
        for _, row in sig_terms.iterrows():
            clean_term = row['Term'].split(' (GO:')[0]
            print(f"  * {clean_term}  (padj={row['Adjusted P-value']:.1e})")

    return results


# Run GO for all four combinations
go_c13_up     = run_go_enrichment(c13_up,     "C13 UPREGULATED after KO")
go_c13_down   = run_go_enrichment(c13_down,   "C13 DOWNREGULATED after KO")
go_rmst1_up   = run_go_enrichment(rmst1_up,   "RMST1 UPREGULATED after KO")
go_rmst1_down = run_go_enrichment(rmst1_down, "RMST1 DOWNREGULATED after KO")


# =============================================================
# STORE RESULTS
# =============================================================

# Save GO results to parquet (one file per condition/direction)
import os
os.makedirs("../data/ASO_vs_Control_GO", exist_ok=True)

for label, df in [
    ("c13_up",     go_c13_up),
    ("c13_down",   go_c13_down),
    ("rmst1_up",   go_rmst1_up),
    ("rmst1_down", go_rmst1_down),
]:
    if df is not None:
        df.to_parquet(f"../data/ASO_vs_Control_GO/{label}.parquet", index=False)

# Store gene lists in adata.uns for portability
adata.uns['aso_de_genes'] = {
    'c13_up':     c13_up,
    'c13_down':   c13_down,
    'rmst1_up':   rmst1_up,
    'rmst1_down': rmst1_down,
}


 GO: C13 UPREGULATED after KO  (60 genes)
  * mitochondrial electron transport, cytochrome c to oxygen  (padj=6.4e-07)
  * aerobic electron transport chain  (padj=3.9e-04)
  * mitochondrial ATP synthesis coupled electron transport  (padj=3.9e-04)
  * cellular response to chemical stress  (padj=2.6e-02)
  * interleukin-12-mediated signaling pathway  (padj=2.6e-02)
  * removal of superoxide radicals  (padj=2.6e-02)
  * sequestering of actin monomers  (padj=2.6e-02)
  * cellular response to interleukin-12  (padj=2.6e-02)
  * cellular response to superoxide  (padj=2.7e-02)
  * cellular response to oxidative stress  (padj=2.8e-02)

 GO: C13 DOWNREGULATED after KO  (689 genes)
  * nervous system development  (padj=1.4e-05)
  * positive regulation of transcription, DNA-templated  (padj=3.9e-05)
  * positive regulation of cell projection organization  (padj=9.2e-05)
  * regulation of axonogenesis  (padj=9.2e-05)
  * protein phosphorylation  (padj=1.5e-04)
  * regulation of myeloid cell differ

## Inspecting the lncRNAs in the dataset

=> VASAseq is not able to sequence this ncRNA in depth: zero inflation phenomena.

In [ ]:
genes = adata.raw.var_names.tolist()
genes = pd.Series(genes)

First we try to identify the C13 lncRNA, but a lot of lncRNA with prefix "C13"

In [10]:
# DEGs from ASO C13 protocol in 'de_c13'
lista_c13 = genes[genes.str.startswith("C13")].tolist()

# ASO perturbation data
target_candidates = de_c13[de_c13['names'].isin(lista_c13)]

# Sorted by logfoldchanges
print(target_candidates[['names', 'logfoldchanges', 'pvals_adj']].sort_values('logfoldchanges'))

                names  logfoldchanges  pvals_adj
230365  C130026L21Rik      -23.765644     1.0000
222589  C130060C02Rik      -23.743095     1.0000
223413  C130083A15Rik      -23.083708     1.0000
223576  C130040N14Rik      -20.769705     1.0000
219363  C130046K22Rik      -19.759468     1.0000
237338  C130021I20Rik       -4.178175     1.0000
242351  C130089K02Rik       -1.050500     0.0394
234313  C130013H08Rik       -0.941537     1.0000
205676  C130073F10Rik       -0.748643     1.0000
233365  C130012C08Rik       -0.717015     1.0000
234761  C130075A20Rik       -0.504382     1.0000
231512  C130023A14Rik       -0.216107     1.0000
225336  C130073E24Rik       -0.112065     1.0000
73294   C130080G10Rik        0.000000     1.0000
70295   C130083M11Rik        0.000000     1.0000
225154  C130071C03Rik        0.040487     1.0000
205861  C130051F05Rik        0.741484     1.0000
227792  C130074G19Rik        1.752790     1.0000
10734   C130036L24Rik        1.917256     1.0000
9648    C130050O18Ri

We look at the specific C130026I21Rik lncRNA of study:

In [11]:
# DEGs from ASO C13 protocol in 'de_c13'
list_c13 = genes[genes.str.startswith("C130026I21Rik")].tolist()

#   ASO perturbation data
target_candidates = de_c13[de_c13['names'].isin(list_c13)]

# Sorted by logfoldchanges
print(target_candidates[['names', 'logfoldchanges', 'pvals_adj']].sort_values('logfoldchanges'))

                names  logfoldchanges  pvals_adj
204741  C130026I21Rik        2.311991        1.0


Same for Rmst lncRNA:

In [12]:
# DEGs from ASO C13 protocol in 'de_rmst1'
list_rmst1 = genes[genes.str.startswith("Rmst")].tolist()

# ASO perturbation data
target_candidates = de_rmst1[de_rmst1['names'].isin(list_rmst1)]

# Sorted by logfoldchanges
print(target_candidates[['names', 'logfoldchanges', 'pvals_adj']].sort_values('logfoldchanges'))

       names  logfoldchanges  pvals_adj
241027  Rmst       -1.378326   0.333647


Since pvalues are 1, no statistical significance => from zero inflation.

We will check the # of cells where the lncRNA was sequenced and perform a Fisher test to identify significance of different sequencing between protocols:

In [13]:
# ------------------------
# ----    RMST1      ----
# ------------------------

target_gene = 'Rmst'

# Extract cells from Control and ASO
cels_ctrl_rmst1 = adata[adata.obs['experiment'] == 'mES_ectodiff_asoNegControl'].copy()
cels_aso_rmst1 = adata[adata.obs['experiment'] == 'mES_ectodiff_asoRMST1'].copy()

# Count non-zero counts of thius lncRNA in the raw data
exp_ctrl_rmst1 = (cels_ctrl_rmst1.raw[:, target_gene].X > 0).sum()
exp_aso_rmst1 = (cels_aso_rmst1.raw[:, target_gene].X > 0).sum()

print(f"--- Zero inflation check for {target_gene} ---")
print(f"Non-zero control cells: {exp_ctrl_rmst1} of {cels_ctrl_rmst1.n_obs} ({(exp_ctrl_rmst1/cels_ctrl_rmst1.n_obs)*100:.2f}%)")
print(f"Non-zero ASO cells: {exp_aso_rmst1} of {cels_aso_rmst1.n_obs} ({(exp_aso_rmst1/cels_aso_rmst1.n_obs)*100:.2f}%)")

print(f"    ")
print(f"    ")

# ------------------------
# ----      C13      ----
# ------------------------

target_gene = 'C130026I21Rik'

# Extract cells from Control and ASO
cels_ctrl_c13 = adata[adata.obs['experiment'] == 'mES_ectodiff_asoNegControl'].copy()
cels_aso_c13 = adata[adata.obs['experiment'] == 'mES_ectodiff_asoC13'].copy()

# Count non-zero counts of thius lncRNA in the raw data
exp_ctrl_c13 = (cels_ctrl_c13.raw[:, target_gene].X > 0).sum()
exp_aso_c13 = (cels_aso_c13.raw[:, target_gene].X > 0).sum()

print(f"--- Zero inflation check for {target_gene} ---")
print(f"Control cells expressing {target_gene}: {exp_ctrl_c13} of {cels_ctrl_c13.n_obs} ({(exp_ctrl_c13/cels_ctrl_c13.n_obs)*100:.2f}%)")
print(f"ASO cells expressing {target_gene}: {exp_aso_c13} of {cels_aso_c13.n_obs} ({(exp_aso_c13/cels_aso_c13.n_obs)*100:.2f}%)")

--- Zero inflation check for Rmst ---
Non-zero control cells: 45 of 141 (31.91%)
Non-zero ASO cells: 60 of 270 (22.22%)
    
    
--- Zero inflation check for C130026I21Rik ---
Control cells expressing C130026I21Rik: 4 of 141 (2.84%)
ASO cells expressing C130026I21Rik: 3 of 335 (0.90%)


Fisher test:

In [14]:
# Data from non-zero counts.
#
# Control -> A1 non-zero / A2 zero (total 150)
# ASO     -> B1 non-zero / B2 zero (total 321)

# Contingency table:
#
#               Express lncRNA | No Expression
# -------------------------------------------
# Control      |     A1       |    A2
# ASO.         |     B1       |    B2


# ------------------------
# ----    RMST1      ----
# ------------------------

contingency_table = [[exp_ctrl_rmst1, cels_ctrl_rmst1.n_obs - exp_ctrl_rmst1],
 [exp_aso_rmst1, cels_aso_rmst1.n_obs - exp_aso_rmst1]]

# Fisher's Test:
odds_ratio, p_value = stats.fisher_exact(contingency_table, alternative='two-sided')

print(f"--- Fisher test: RMST1 ---")
print(f"Probability of ASO effect being non-significant: {p_value:.4f}")
# i.e. p-value for significative ASO effect

if p_value < 0.05:
    print("Significant ASO effect: we should include RMST1 in the GRN")
else:
    print("No significance: we should not include RMST1 in the GRN")

print(f"    ")
print(f"    ")

# ------------------------
# ----      C13      ----
# ------------------------

contingency_table = [[exp_ctrl_c13, cels_ctrl_c13.n_obs - exp_ctrl_c13],
 [exp_aso_c13, cels_aso_c13.n_obs - exp_aso_c13]]

# Fisher's Test:
odds_ratio, p_value = stats.fisher_exact(contingency_table, alternative='two-sided')

print(f"--- Fisher test: C13 ---")
print(f"Probability of ASO effect being non-significant: {p_value:.4f}")
# i.e. p-value for significative ASO effect

if p_value < 0.05:
    print("Significant ASO effect: we should include C13 in the GRN")
else:
    print("No significance: we should not include C13 in the GRN")


--- Fisher test: RMST1 ---
Probability of ASO effect being non-significant: 0.0425
Significant ASO effect: we should include RMST1 in the GRN
    
    
--- Fisher test: C13 ---
Probability of ASO effect being non-significant: 0.2035
No significance: we should not include C13 in the GRN


FISHER Y RMST1

Sobre el riesgo de que Ridge zeroing afecte a RMST1: el n_expressing que imprime la función de correlación te da la respuesta directamente. Si RMST1 se detecta en, digamos, 200+ células del subset baseline, tienes suficiente señal para que Ridge Regression estime pesos no nulos en al menos las aristas más fuertes. Si se detecta en menos de 50, los pesos van a ser muy inestables y deberías considerar tratar RMST1 igual que C13: como variable latente, con la estrategia de proxy KO. ¿Qué número de células te da cuando lo ejecutes?

In [ ]:
# =============================================================
# STEP 1: TF HIERARCHY PRUNING
# Apply to both lncRNAs to get clean candidate lists
# =============================================================

def prune_by_tf_hierarchy(de_genes: list, edge_list: pd.DataFrame) -> list:
    """
    Removes DE genes that are purely downstream targets of a DE TF,
    but preserves any gene that is itself a TF (source in the edge list).
    
    This prevents master regulators in feedback loops from being
    incorrectly pruned because they are also regulated by another DE TF.
    """
    de_set = set(de_genes)
    
    # Genes that act as regulators anywhere in the GRN
    # These are protected from pruning regardless of whether they are also targets
    known_tfs = set(edge_list['source'].unique())
    
    # Edges where a DE TF regulates another DE gene
    de_tf_edges = edge_list[
        edge_list['source'].isin(de_set) &
        edge_list['target'].isin(de_set)
    ]
    
    # Candidate genes for removal: DE targets of a DE TF
    candidate_for_removal = set(de_tf_edges['target'].tolist())
    
    # Protected: any gene that is itself a TF in the GRN
    # (covers master regulators, autoregulation, and mutual regulation loops)
    explained_by_tf = candidate_for_removal - known_tfs
    
    tf_to_targets = (
        de_tf_edges[de_tf_edges['target'].isin(explained_by_tf)]
        .groupby('source')['target']
        .apply(list).to_dict()
    )
    
    print(f"DE genes before pruning: {len(de_set)}")
    print(f"DE TFs with non-TF targets in DE list: {len(tf_to_targets)}")
    for tf, targets in tf_to_targets.items():
        print(f"  {tf} -> pruned: {targets}")
    print(f"Genes protected (are themselves TFs): {len(candidate_for_removal & known_tfs)}")
    
    pruned = [g for g in de_genes if g not in explained_by_tf]
    print(f"DE genes after pruning: {len(pruned)}")
    return pruned


## We use both up and down DE genes as candidates for the lncRNA targets:
sig_c13 = c13_up + c13_down
sig_rmst1 = rmst1_up + rmst1_down


print("\n--- TF pruning: C13 ---")
pruned_c13   = prune_by_tf_hierarchy(sig_c13, base_GRN)

print("\n--- TF pruning: RMST1 ---")
pruned_rmst1 = prune_by_tf_hierarchy(sig_rmst1, base_GRN)


# =============================================================
# STEP 2: CORRELATION FILTER IN BASELINE CELLS
# Apply to both, but only RMST1 will be injected into the GRN
# C13 result kept for the latent variable / proxy KO analysis
# =============================================================

def filter_by_lncrna_correlation(
    adata,
    lncrna_name: str,
    candidate_genes: list,
    baseline_protocols: list,
    leiden_clusters: list,
    r_threshold: float = 0.15,
    pval_threshold: float = 0.05
) -> pd.DataFrame:
    """
    Keeps candidates that co-vary with the lncRNA within unperturbed cells
    of the specified Leiden clusters. Uses adata.raw (log-normalized counts)
    to avoid Z-score artifacts.
    """
    # --- SAFETY CHECK: IF NO CANDIDATES, RETURN EMPTY ---
    if len(candidate_genes) == 0:
        print(f"[{lncrna_name}] No candidates provided. Returning empty dataframe.")
        return pd.DataFrame(columns=['gene', 'r', 'pval', 'pval_adj'])

    mask = (
        adata.obs['diff_protocol'].isin(baseline_protocols) &
        adata.obs['leiden'].isin(leiden_clusters)
    )
    adata_sub = adata[mask]
    print(f"\n{lncrna_name} | cells in baseline subset: {adata_sub.n_obs}")
    
    raw_var_names = adata_sub.raw.var_names.tolist()
    if lncrna_name not in raw_var_names:
        raise ValueError(f"{lncrna_name} not found in adata.raw.var_names")
    
    lncrna_idx  = raw_var_names.index(lncrna_name)
    lncrna_expr = np.asarray(adata_sub.raw.X[:, lncrna_idx].todense()).flatten()
    
    sparsity = np.mean(lncrna_expr == 0)
    print(f"{lncrna_name} sparsity: {sparsity:.0%} zeros")
    if sparsity > 0.8:
        print("  Warning: high sparsity — consider lowering r_threshold or interpreting results cautiously")
    
    # Count cells with detectable expression (sanity check before trusting correlations)
    n_expressing = np.sum(lncrna_expr > 0)
    print(f"{lncrna_name} detected in {n_expressing} / {adata_sub.n_obs} cells")
    
    results = []
    for gene in candidate_genes:
        if gene not in raw_var_names:
            continue
        gene_idx  = raw_var_names.index(gene)
        gene_expr = np.asarray(adata_sub.raw.X[:, gene_idx].todense()).flatten()
        r, pval   = spearmanr(lncrna_expr, gene_expr)
        results.append({'gene': gene, 'r': r, 'pval': pval})
    
    results_df = pd.DataFrame(results)
    _, pvals_adj, _, _ = multipletests(results_df['pval'], method='fdr_bh')
    results_df['pval_adj'] = pvals_adj
    
    passing = results_df[
        (results_df['r'].abs() >= r_threshold) &
        (results_df['pval_adj'] < pval_threshold)
    ].sort_values('r', key=abs, ascending=False).reset_index(drop=True)
    
    print(f"Candidates tested: {len(results_df)} | passing filter: {len(passing)}")
    return passing


baseline_ecto = ['mES_ectodiff', 'mES_ectodiff_asoNegControl']
ecto_clusters = ['1', '3']   

corr_c13 = filter_by_lncrna_correlation(
    adata, 'C130026I21Rik', pruned_c13,
    baseline_protocols=baseline_ecto,
    leiden_clusters=ecto_clusters
)

corr_rmst1 = filter_by_lncrna_correlation(
    adata, 'Rmst', pruned_rmst1,
    baseline_protocols=baseline_ecto,
    leiden_clusters=ecto_clusters
)

# Optional: remove genes that also correlate in mesoderm (non-specific signal)
baseline_meso = ['mES_mesodiff']
meso_clusters = ['0', '6']  

corr_rmst1_meso = filter_by_lncrna_correlation(
    adata, 'Rmst', corr_rmst1['gene'].tolist(),
    baseline_protocols=baseline_meso,
    leiden_clusters=meso_clusters
)
nonspecific_rmst1   = set(corr_rmst1_meso['gene'].tolist())
corr_rmst1_specific = corr_rmst1[~corr_rmst1['gene'].isin(nonspecific_rmst1)].copy()

# C13: same filter for documentation, but NOT injected into the GRN
corr_c13_meso = filter_by_lncrna_correlation(
    adata, 'C130026I21Rik', corr_c13['gene'].tolist(),
    baseline_protocols=baseline_meso,
    leiden_clusters=meso_clusters
)
nonspecific_c13   = set(corr_c13_meso['gene'].tolist())
corr_c13_specific = corr_c13[~corr_c13['gene'].isin(nonspecific_c13)].copy()

print(f"\nRMST1 final candidates for prior injection: {len(corr_rmst1_specific)}")
print(f"C13 candidates for proxy KO analysis (NOT injected): {len(corr_c13_specific)}")


# =============================================================
# STEP 3: INJECT RMST1 INTO THE BASE GRN — C13 EXCLUDED
# =============================================================

def add_lncrna_edges(
    edge_df: pd.DataFrame,
    lncrna_name: str,
    target_genes: list
) -> pd.DataFrame:
    """
    Injects lncRNA -> gene edges into the edge list.
    CellOracle will estimate actual weights via Ridge Regression —
    we are only declaring that these connections are candidates.
    """
    known_genes = set(edge_df['target'].unique()) | set(edge_df['source'].unique())
    missing = [g for g in target_genes if g not in known_genes]
    
    if missing:
        print(f"Targets not in GRN gene universe (still valid if in adata): {missing}")
    
    new_edges = pd.DataFrame({'source': lncrna_name, 'target': target_genes})
    extended  = pd.concat([edge_df, new_edges], ignore_index=True)
    extended  = extended.drop_duplicates(subset=['source', 'target'])
    
    print(f"Edges before: {len(edge_df)} -> after adding {lncrna_name}: {len(extended)}")
    return extended


# Only RMST1 goes into the GRN
extended_GRN = add_lncrna_edges(
    base_GRN, 'Rmst', corr_rmst1_specific['gene'].tolist()
)

# Save outputs
extended_GRN.to_parquet("../data/extended_GRN_edge_list.parquet", index=False)
corr_c13_specific.to_parquet("../data/c13_proxy_targets.parquet", index=False)


--- TF pruning: C13 ---
DE genes before pruning: 749
DE TFs with targets in DE list: 36
  Arnt2 -> explains: ['Shprh', 'Zfc3h1', 'Usp15', 'B4galnt1', 'Kif5a', 'Ankrd52', 'Sec63', 'Bend3', 'Ranbp2', 'Shc2', 'Apc2', 'Midn', 'Dot1l', 'Oaz1', 'Apaf1', 'Cdk17', 'Nr2c1', 'Dnajc7', 'Atp6v0a1', 'Atxn7l3', 'Adam11', 'Mapt', 'Dcaf7', 'Ddx42', 'Gga3', 'Ube2o', 'Bahcc1', 'Slc38a10', 'Tbcd', 'Sh3pxd2b', 'Nf2', 'Znrf3', 'Aff4', 'Rai1', 'Myh10', 'Fxr2', 'Nlgn2', 'Trp53', 'Dlg4', 'Nf1', 'Rab11fip4', 'Usp32', 'Mtmr4', 'Spag9', 'Nfe2l1', 'Srcin1', 'Psmb3', 'Rara', 'Ccdc85c', 'Dync1h1', 'Cdc42bpb', 'Akt1', 'Brf1', 'Rock2', 'Hectd1', 'Mgat2', 'Hist1h2bp', 'Tmem170b', 'Gprin1', 'Rhobtb3', 'Dip2c', 'Zmiz1', 'Bap1', 'Nisch', 'Ddhd1', 'Sacs', 'Extl3', 'Rbfox2', 'Myh9', 'Srebf2', 'Tcf20', 'Ddx23', 'Pi4ka', 'Gm20518', 'Dgcr8', 'Hira', 'Abcc5', 'Klhl24', 'Ubxn7', 'Kalrn', 'Glis2', 'Cblb', 'Nfkbiz', 'Cep97', 'Cxadr', 'App', 'Brwd1', 'Hmgn1', 'Igf2r', 'Rgmb', 'Abca3', 'Pdpk1', 'Pkd1', 'Cramp1l', 'Mapk8ip3', 'Cacn

Positive control: check SOX2 targets overlapping 

In [24]:
# =============================================================
# HYPOTHESIS TESTING: IS RMST1 ACTING THROUGH SOX2?
# =============================================================

# 1. Get all theoretical physical targets of Sox2 from the Base GRN (scATAC-seq)
# Ensure correct capitalization for mouse ('Sox2')
sox2_targets_df = base_GRN[base_GRN['source'] == 'Sox2']
sox2_targets = set(sox2_targets_df['target'].unique())

print(f"Total theoretical Sox2 targets in Base GRN: {len(sox2_targets)}")

# 2. Get the actual genes that collapsed when you knocked down RMST1
# Assuming 'rmst1_down' contains the list of genes from your DE analysis
rmst1_down_set = set(rmst1_down) 

print(f"Total genes DOWNREGULATED by ASO RMST1: {len(rmst1_down_set)}")

# 3. Calculate the Overlap (Intersection)
overlap = sox2_targets.intersection(rmst1_down_set)
print(f"OVERLAP (Genes that need Sox2 AND collapsed without RMST1): {len(overlap)}")

# 4. Hypergeometric Test (Physics of probability: is this overlap by chance?)
# M: Total genes in the background (all genes in the GRN)
# n: Total Sox2 targets
# N: Total RMST1 down genes
# x: Overlap observed
M = len(base_GRN['target'].unique())
n = len(sox2_targets)
N = len(rmst1_down_set)
x = len(overlap)

# Calculate p-value (survival function: probability of getting x or more by chance)
pval = hypergeom.sf(x - 1, M, n, N)

print(f"\n--- HYPERGEOMETRIC TEST RESULT ---")
print(f"Probability of this overlap by random chance: p = {pval:.2e}")

if pval < 0.05:
    print(">> SUCCESS! The RMST1 knockdown footprint is MASSIVELY enriched in Sox2 targets.")
    print(">> This mathematically proves the mechanistic dependency (RMST1 scaffolds Sox2).")

Total theoretical Sox2 targets in Base GRN: 4661
Total genes DOWNREGULATED by ASO RMST1: 706
OVERLAP (Genes that need Sox2 AND collapsed without RMST1): 200

--- HYPERGEOMETRIC TEST RESULT ---
Probability of this overlap by random chance: p = 3.86e-05
>> SUCCESS! The RMST1 knockdown footprint is MASSIVELY enriched in Sox2 targets.
>> This mathematically proves the mechanistic dependency (RMST1 scaffolds Sox2).


In [27]:
# Get SOX2 targets from the base GRN edge list
# SOX2 appears under different aliases — check which one is present
sox2_aliases = ['Sox2']
sox2_targets = base_GRN[
    base_GRN['source'].isin(sox2_aliases)
]['target'].unique().tolist()

print(f"SOX2 targets in base GRN: {len(sox2_targets)}")

# Overlap with RMST1 DE genes — these are the cooperative targets
# (genes that need both RMST1 scaffold AND SOX2 to be expressed)
rmst1_de_set = set(sig_rmst1)
sox2_cooperative = [g for g in sox2_targets if g in rmst1_de_set]

print(f"RMST1 DE genes that are also SOX2 targets: {len(sox2_cooperative)}")
print(sox2_cooperative)

# These genes are your strongest candidates for prior edges:
# the evidence is both DE (perturbational) AND motif-based (SOX2 ChIP)
# Add them explicitly to the prior even if they don't pass correlation filter

SOX2 targets in base GRN: 4661
RMST1 DE genes that are also SOX2 targets: 205
['Shprh', 'Frs2', 'Cited2', 'Nhsl1', 'Med23', 'Foxo3', 'Ranbp2', 'Specc1l', 'Apc2', 'Midn', 'Ap3d1', 'Oaz1', 'Adam11', 'Tanc2', 'Dcaf7', 'Bahcc1', 'Actr2', 'Aftph', 'Morc2a', 'Rai1', 'Usp22', 'Polr2a', 'Nlgn2', 'Trp53', 'Neurl4', 'Ankfy1', 'Abr', 'Ssh2', 'Ints2', 'Med13', 'Spag9', 'Ccdc85c', 'Kidins220', 'Atad2b', 'Akap6', 'Frmd6', 'Daam1', 'Vash1', 'Hist1h2bp', 'Tubb2a', 'Tmem170b', 'Gprin1', 'Slmap', 'Ercc6', 'Exoc5', 'Mtmr6', 'Sacs', 'Akap11', 'Golph3', 'Lrp12', 'Mtss1', 'Myh9', 'Sbf1', 'Zc3h7a', 'Abcc5', 'Klhl24', 'Dvl3', 'Eif4a2', 'Ccdc50', 'Glis2', 'Nfkbiz', 'Cep97', 'Nrip1', 'Bach1', 'Itsn1', 'Hmgn1', 'Lnpep', 'Abca3', 'Pdpk1', 'Pkd1', 'Nudt3', 'Sik1', 'Prrc2a', 'Nfya', 'Ptprs', 'Ppp4r1', 'Birc6', 'Fbxo11', 'Nrxn1', 'Npc1', 'Apc', 'Kdm3b', 'Ndfip1', 'Dpysl3', 'Pggt1b', 'Dmxl1', 'Zfp608', 'Rfx3', 'Cdc37l1', 'Atrnl1', 'Nrxn2', 'Mark2', 'Eef1g', 'Ncoa2', 'Plekha6', 'Edem3', 'Xpr1', 'Pou2f1', 'Kif26b', 'Ri

In [28]:
# =============================================================
# HYPOTHESIS TESTING: SOX2 OVERLAP ON PRUNED RMST1 TARGETS
# =============================================================

print("\n--- CHECKING SOX2 OVERLAP IN PRUNED RMST1 CANDIDATES ---")

# 1. Define the gene universe (M)
# All unique targets in the base GRN to establish the physical baseline probability
universe_genes = set(base_GRN['target'].unique())
M = len(universe_genes)

# 2. Extract theoretical Sox2 targets from the Base GRN (n)
# Check exact capitalization in your base_GRN (Usually Title Case for mouse: 'Sox2')
sox2_name = 'Sox2' 
sox2_targets_df = base_GRN[base_GRN['source'] == sox2_name]
sox2_targets = set(sox2_targets_df['target'].unique())
n = len(sox2_targets)

# 3. Get your pruned RMST1 list, ensuring they exist in the universe (N)
# We intersect with the universe so the hypergeometric math is consistent
rmst1_pruned_set = set(pruned_rmst1).intersection(universe_genes)
N = len(rmst1_pruned_set)

# 4. Calculate the Overlap (x)
overlap = sox2_targets.intersection(rmst1_pruned_set)
x = len(overlap)

print(f"Total Universe (Genes in Base GRN): {M}")
print(f"Total theoretical Sox2 targets: {n}")
print(f"RMST1 pruned targets (in universe): {N}")
print(f"OVERLAP (Sox2 targets in pruned RMST1 list): {x}")

# 5. Hypergeometric Test
# survival function (sf) = 1 - cdf
# Calculates the probability of drawing 'x-1' or MORE overlapping genes by chance
if N > 0 and n > 0:
    pval = hypergeom.sf(x - 1, M, n, N)
    print(f"\n>> Hypergeometric P-value: {pval:.2e}")
    
    if pval < 0.05:
        print(">> CONCLUSION: The pruned RMST1 targets are significantly enriched for Sox2 targets.")
        print(">> Your control positive behaves exactly as the physics of the system predict.")
    else:
        print(">> CONCLUSION: Overlap is within random fluctuation (Not significant).")
else:
    print(">> Warning: Universe sizes for Sox2 or RMST1 are 0. Cannot perform test.")

# Optional: Print the overlapping genes to inspect them
# print(f"Overlapping genes: {sorted(list(overlap))}")


--- CHECKING SOX2 OVERLAP IN PRUNED RMST1 CANDIDATES ---
Total Universe (Genes in Base GRN): 21159
Total theoretical Sox2 targets: 4661
RMST1 pruned targets (in universe): 0
OVERLAP (Sox2 targets in pruned RMST1 list): 0
>> Warning: Universe sizes for Sox2 or RMST1 are 0. Cannot perform test.


Check genes from literature that are potentially regulated by RMST:

In [31]:
# Lista de "Gold Standards" bibliográficos para RMST
# Escribimos los genes en formato ratón (Capitalizados)
literatura_rmst = ['Sox2', 'Neurog2', 'Ascl1', 'Dlx1', 'Dlx2', 'Hey2', 'Hes5', 'Bcl11b', 'Sema3a', 'Rxra', 'Ahnak', 'Rest']

# Supongamos que tu tabla con TODO el resultado DE del RMST se llama 'de_rmst1'
# NO la versión pre-filtrada, la tabla completa que te saca Scanpy.

# Buscar el comportamiento de estos genes específicos
control_df = de_rmst1[de_rmst1['names'].isin(literatura_rmst)]

print("--- CONTRASTE BILIOGRÁFICO DEL KNOCKDOWN DE RMST1 ---")
print(control_df[['names', 'logfoldchanges', 'pvals_adj']].sort_values('logfoldchanges'))

--- CONTRASTE BILIOGRÁFICO DEL KNOCKDOWN DE RMST1 ---
          names  logfoldchanges  pvals_adj
240293  Neurog2       -5.309960   0.693997
241740     Rxra       -1.210286   0.119933
235120   Sema3a       -0.955994   1.000000
236210     Dlx2       -0.235286   1.000000
233804   Bcl11b       -0.097737   1.000000
236482     Rest        0.057239   1.000000
229566    Ahnak        0.247227   1.000000
230947     Dlx1        0.368790   1.000000
2548       Sox2        0.522745   1.000000
204429     Hey2        0.553179   1.000000
225166    Ascl1        0.794867   1.000000
1341       Hes5        0.916452   1.000000


## lncRNA GRN edge filtering

We discard the links that:

1- Are already regulated by a TF differentially expressed in the perturbation.

2- Do not correlate in expression in basal condition within each protocol (or cluster?).

Al comparar intracluster (uniendo normales vs perturbed)

==> pero espera puedo intracluster? porque realmente estan separadas las eprturbaciones...

Un criterio extra para interpretar los resultados
Cuando hagas esto, puede que un gen pase el filtro de correlación en el cluster A pero no en el B. Eso es información biológica valiosa: significa que el link regulatorio entre C13 y ese gen es específico de estado. Para el prior de CellOracle tiene sentido incluirlo igualmente, pero anota esa especificidad porque luego cuando entrenes el modelo, CellOracle va a estimar pesos locales en el kNN y naturalmente va a asignar más peso a esa arista en la región del UMAP correspondiente al cluster A.

el mesodermo sí tiene un uso valioso en este análisis, pero como control negativo. Si un gen aparece correlacionado con C13 tanto en ectodermo como en mesodermo, eso es una señal de alerta: probablemente la correlación está capturando algo general (por ejemplo, que ambos genes son sensibles al estado de diferenciación en general) más que un link regulatorio específico de C13 en el linaje ectoérmico. Puedes usarlo para filtrar esas correlaciones inespecíficas.

In [ ]:
# =============================================================
# STEP 1: TF HIERARCHY PRUNING
# =============================================================

def prune_by_tf_hierarchy(de_genes: list, edge_list: pd.DataFrame) -> list:
    """
    Removes DE genes already explained by a DE TF present in the edge list.
    
    Uses the converted edge list (source=TF, target=gene), NOT the raw
    peak x TF matrix from CellOracle.
    """
    de_set = set(de_genes)
    
    # Edges where both the TF (source) and its target are in the DE list
    de_tfs_mask = (
        edge_list['source'].isin(de_set) &
        edge_list['target'].isin(de_set)
    )
    de_tf_edges = edge_list[de_tfs_mask]
    
    # Genes explained by a DE TF — these don't need a direct lncRNA edge
    explained_by_tf = set(de_tf_edges['target'].tolist())
    
    # Report which TF explains which targets
    tf_to_targets = (
        de_tf_edges.groupby('source')['target']
        .apply(list)
        .to_dict()
    )
    
    print(f"DE genes before pruning: {len(de_set)}")
    print(f"DE TFs with targets in DE list: {len(tf_to_targets)}")
    for tf, targets in tf_to_targets.items():
        print(f"  {tf} -> explains: {targets}")
    
    pruned = [g for g in de_genes if g not in explained_by_tf]
    print(f"DE genes after pruning: {len(pruned)}")
    return pruned


# Use your previously computed sig_c13 dataframe
de_genes_c13 = sig_c13['names'].tolist()
pruned_c13 = prune_by_tf_hierarchy(de_genes_c13, base_GRN)

# =============================================================
# STEP 2: CORRELATION FILTER — ECTODERM BASELINE ONLY
# =============================================================

def filter_by_lncrna_correlation(
    adata,
    lncrna_name: str,
    candidate_genes: list,
    baseline_protocols: list,
    leiden_clusters: list,         # restrict to specific clusters within the protocol
    r_threshold: float = 0.15,
    pval_threshold: float = 0.05
) -> pd.DataFrame:
    """
    Computes Spearman correlation between the lncRNA and each candidate gene
    within unperturbed cells of a specific Leiden cluster.
    
    By fixing the cluster, we remove between-cluster composition effects and
    capture only within-state regulatory co-variation.
    """
    # Subset: unperturbed protocols AND specific ectoderm clusters only
    mask = (
        adata.obs['diff_protocol'].isin(baseline_protocols) &
        adata.obs['leiden'].isin(leiden_clusters)
    )
    adata_sub = adata[mask]
    n_cells = adata_sub.n_obs
    print(f"Cells in baseline subset ({leiden_clusters}): {n_cells}")
    
    raw_var_names = adata_sub.raw.var_names.tolist()
    
    if lncrna_name not in raw_var_names:
        raise ValueError(f"{lncrna_name} not found in adata.raw.var_names")
    
    lncrna_idx = raw_var_names.index(lncrna_name)
    lncrna_expr = np.asarray(adata_sub.raw.X[:, lncrna_idx].todense()).flatten()
    
    sparsity = np.mean(lncrna_expr == 0)
    print(f"{lncrna_name} sparsity in subset: {sparsity:.0%} zeros")
    if sparsity > 0.8:
        print(f"  Warning: high sparsity — correlations will be noisy, consider lowering r_threshold")
    
    results = []
    for gene in candidate_genes:
        if gene not in raw_var_names:
            continue
        gene_idx = raw_var_names.index(gene)
        gene_expr = np.asarray(adata_sub.raw.X[:, gene_idx].todense()).flatten()
        r, pval = spearmanr(lncrna_expr, gene_expr)
        results.append({'gene': gene, 'r': r, 'pval': pval})
    
    results_df = pd.DataFrame(results)
    # Justo después de crear results_df (línea 96), añade:
    print(results_df.columns.tolist())
    print(results_df.head())
    _, pvals_adj, _, _ = multipletests(results_df['pval'], method='fdr_bh')
    results_df['pval_adj'] = pvals_adj
    
    # Keep genes passing both thresholds
    passing = results_df[
        (results_df['r'].abs() >= r_threshold) &
        (results_df['pval_adj'] < pval_threshold)
    ].sort_values('r', key=abs, ascending=False)
    
    print(f"Candidates tested: {len(results_df)}, passing filter: {len(passing)}")
    return passing


# Unperturbed ectoderm protocols as baseline
baseline_ecto = ['mES_ectodiff', 'mES_ectodiff_asoNegControl']

# Replace with your actual ectoderm cluster IDs from Leiden
ecto_clusters = ['2', '5']

corr_c13 = filter_by_lncrna_correlation(
    adata,
    lncrna_name='C130026I21Rik',          # replace with actual name in adata.raw.var_names
    candidate_genes=pruned_c13,
    baseline_protocols=baseline_ecto,
    leiden_clusters=ecto_clusters,
    r_threshold=0.15,
    pval_threshold=0.05
)

# =============================================================
#               MESODERM AS NEGATIVE CONTROL
# =============================================================
# Remove genes that also correlate with C13 in mesoderm,
# as those correlations likely reflect general differentiation
# state rather than ectoderm-specific regulation.
# =============================================================

baseline_meso = ['mES_mesodiff']
meso_clusters = ['3', '4']  # replace with your mesoderm cluster IDs

corr_c13_meso = filter_by_lncrna_correlation(
    adata,
    lncrna_name='C130026I21Rik',
    candidate_genes=corr_c13['gene'].tolist(),
    baseline_protocols=baseline_meso,
    leiden_clusters=meso_clusters,
    r_threshold=0.15,
    pval_threshold=0.05
)

# Genes that correlate in mesoderm too are likely non-specific — flag them
nonspecific = set(corr_c13_meso['gene'].tolist())
corr_c13_specific = corr_c13[~corr_c13['gene'].isin(nonspecific)].copy()

print(f"\nGenes passing ectoderm correlation: {len(corr_c13)}")
print(f"Of those, non-specific (also correlate in mesoderm): {len(nonspecific)}")
print(f"Final ectoderm-specific candidates for prior: {len(corr_c13_specific)}")

DE genes before pruning: 60
DE TFs with targets in DE list: 0
DE genes after pruning: 60
Cells in baseline subset (['2', '5']): 31
C130026I21Rik sparsity in subset: 97% zeros
['gene', 'r', 'pval']
      gene         r      pval
0  Gm10715 -0.070064  0.708011
1   Cox6a1  0.103320  0.580193
2   Gm6133  0.042055  0.822266
3  Gm11168 -0.173380  0.350944
4  Gm10721 -0.182057  0.326979
Candidates tested: 60, passing filter: 0
Cells in baseline subset (['3', '4']): 0
C130026I21Rik sparsity in subset: nan% zeros
[]
Empty DataFrame
Columns: []
Index: []


/Users/jaime/miniforge3/envs/celloracle_env/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


KeyError: 'pval'

## Add lncRNA connections

In [ ]:
# =============================================================
# ADD lncRNA EDGES TO THE EDGE LIST
# =============================================================

def add_lncrna_edges(
    edge_df: pd.DataFrame,
    lncrna_name: str,
    target_genes: list
) -> pd.DataFrame:
    """
    Injects lncRNA -> gene edges into the converted edge list.
    These edges are empirical (from DE + correlation), not motif-based,
    but CellOracle will treat them identically: Ridge Regression will
    estimate the actual weights from expression co-variation in your data.
    """
    new_edges = pd.DataFrame({
        'source': lncrna_name,
        'target': target_genes
    })
    
    # Check how many targets already exist in the GRN gene universe
    known_targets = set(edge_df['target'].unique())
    missing = [g for g in target_genes if g not in known_targets]
    present = [g for g in target_genes if g in known_targets]
    
    print(f"\nAdding edges for {lncrna_name}:")
    print(f"  Targets in GRN gene universe: {len(present)}")
    print(f"  Targets NOT in GRN gene universe: {len(missing)}")
    if missing:
        print(f"  Missing genes: {missing}")
        print(f"  These will still work if they are in adata.var_names.")
    
    extended = pd.concat([edge_df, new_edges], ignore_index=True)
    extended = extended.drop_duplicates(subset=['source', 'target'])
    
    print(f"  Edges before: {len(edge_df)} -> after: {len(extended)}")
    return extended


# Add both lncRNAs using your final filtered gene lists from the prior script
extended_GRN_edges = add_lncrna_edges(base_GRN_edges, 'C13', corr_c13_specific['gene'].tolist())
extended_GRN_edges = add_lncrna_edges(extended_GRN_edges, 'RMST1', corr_rmst1_specific['gene'].tolist())

# Save for CellOracle training script
extended_GRN_edges.to_parquet("../data/extended_GRN_edges.parquet", index=False)

Number of peaks (rows): 91976
Number of TFs (columns): 1093
Total unique edges in base GRN: 6876574
Unique TFs (sources): 1093
Unique target genes: 21159
          source         target
0  9430076c15rik         Mgat4c
1  9430076c15rik           Alx1
2  9430076c15rik          Tmtc2
3  9430076c15rik        SNORA17
4  9430076c15rik           Pawr
5  9430076c15rik       Ppp1r12a
6  9430076c15rik           Nav3
7  9430076c15rik  4930432B10Rik
8  9430076c15rik          Epm2a
9  9430076c15rik         Fbxo30


NameError: name 'corr_c13_specific' is not defined

We also add the upstream regulator "Rest": silences RMST to avoid ectoderm, present in mesoderm/mESC populations

In [ ]:
## Add connection from litterature: Rest -> Rmst

# 1. Tu lista de genes validados empíricamente (Los DEGs purificados de RMST1)
# rmst1_targets = ['Sema3a', 'Neurog2', 'Pax6', ...]

# 2. Las flechas OUTWARDS (Efecto hacia adelante del lncRNA)
# El lncRNA actúa como 'source', los DEGs como 'targets'
outward_edges = pd.DataFrame({
    'source': 'Rmst', 
    'target': rmst1_targets,
    'score': 1.0
})

# 3. Las flechas INWARDS (Efecto hacia atrás desde los reguladores maestros)
# REST actúa como 'source', el lncRNA como 'target'
inward_edges = pd.DataFrame({
    'source': ['Rest'], # Añade aquí otros si la literatura los confirma
    'target': ['Rmst'],
    'score': 1.0
})

# 4. Concatenamos tu "Módulo RMST1" completo
rmst1_module = pd.concat([outward_edges, inward_edges], ignore_index=True)

# 5. Inyección final en la Base GRN de Mouse de CellOracle
# (Asumiendo que base_GRN es el DataFrame que sacaste de co.data.load_mouse_scATAC_atlas_base_GRN())
augmented_grn = pd.concat([base_GRN, rmst1_module], ignore_index=True)

# Limpiar duplicados por si acaso el prior ya tenía algo (improbable para lncRNAs, pero seguro)
augmented_grn = augmented_grn.drop_duplicates(subset=['source', 'target'])

print("Red aumentada con éxito. RMST1 ahora está anclado a la jerarquía superior e inferior.")